# BVA medallion ingest (monolithic)

| Field | Value |
| ----- | ----- |
| **Sprint** | Sprint 15 T3 published + Sprint 15.1 mini-sprint close (S15.1 ✅) |
| **Runtime** | Microsoft Fabric notebook (PySpark, runtime 1.3) |
| **Trigger** | `.github/workflows/bva-sim-refresh.yml` via `POST /items/{nb}/jobs/instances?jobType=RunNotebook` |
| **Input** | `Files/Bronze/consumption/BillingPeriod=*/ChargePeriodStart=*/part-00000.parquet` (uploaded by `bva_upload_bronze.py`) |
| **Output** | `bronze.bva_consumption` + `silver.bva_consumption` + 8 `gold.bva_dim_*` + 3 `gold.bva_fact_*` Delta tables |
| **Contract** | `docs/superpowers/specs/2026-07-09-sprint-15-bva-design.md` §5 (star schema) |
| **Source of truth** | `data-platform/notebooks/bva/bva_transforms.py` (pure module, unit-tested by `data-platform/notebooks/bva/tests/`). Inlined into Cell 5 below because Fabric notebooks can't `import` repo-local `.py` without a custom-library upload; established repo pattern (`csa-verify-mvp`) uses inline. |

Runs the 4 medallion stages sequentially:

1. **Bronze register** — glob-read the daily parquet partitions the workflow uploaded, write as Delta `bronze.bva_consumption`.
2. **Silver** — normalise FOCUS rows to the Silver contract (foreign keys, provenance).
3. **Gold dims** — 8 dimension tables (service, meter, resource, environment, hospital, capability, date, exec_role).
4. **Gold facts** — 3 fact tables (azure_consumption, budget, value_realization).

**Design note (Sprint 15.1).** The `bva-sim-refresh.yml` workflow was originally
designed to trigger a Fabric Data Factory Pipeline (`?jobType=Pipeline`). During
Sprint 15.1 we pivoted to a monolithic notebook + `?jobType=RunNotebook` — the
same shape as `adoption-refresh.yml` (validated end-to-end on 2026-07-15) — to
avoid the least-documented Fabric REST surface (Data Factory Pipeline authoring)
and get to a green medallion faster. Splitting into 4 per-stage notebooks +
Pipeline can happen later if per-stage retry granularity is needed.

**No PHI.** Synthetic FOCUS data only; adoption sign-in join redacts IPs to /24
at the Bronze layer (`adoption_transforms.redact_ip_24`).


## 0. Schema pre-creation

Fabric lakehouse tables named `bronze.foo` / `silver.foo` / `gold.foo` require
the corresponding schema/database to exist before `.saveAsTable(...)`.
Idempotent (`CREATE SCHEMA IF NOT EXISTS`).


In [ ]:
for _schema in ('bronze', 'silver', 'gold'):
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {_schema}')
    print(f'schema ok: {_schema}')


## 1. Bronze register (parquet → Delta table)


In [ ]:
# The workflow's bva_upload_bronze.py lands partitions at:
#   Files/Bronze/consumption/BillingPeriod=YYYY-MM/ChargePeriodStart=YYYY-MM-DD/part-00000.parquet
_BRONZE_PARQUET_GLOB = 'Files/Bronze/consumption/BillingPeriod=*/ChargePeriodStart=*/part-00000.parquet'
_bronze_df = spark.read.parquet(_BRONZE_PARQUET_GLOB)

(
    _bronze_df.write.format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('bronze.bva_consumption')
)
print(f'bronze: wrote bronze.bva_consumption ({_bronze_df.count()} rows)')


## 2. Inlined `bva_transforms`

Source of truth: [`data-platform/notebooks/bva/bva_transforms.py`](../bva_transforms.py) — pure, unit-tested.

When the repo module changes, this cell must be re-synced. A CI drift-check
is tracked as a Sprint 15.1 follow-up.


In [ ]:
# --- BEGIN INLINED bva_transforms.py ---
# Source of truth: data-platform/notebooks/bva/bva_transforms.py
from typing import Iterable, Mapping

# --------------------------------------------------------------------------- #
# Synthetic value-realization calibration (design spec §5–§6).
#
# These are *synthetic* rates used to derive plan-vs-actual value KPIs from the
# FOCUS consumption seed. They are deliberately simple and documented so the
# derived KPIs stay explainable in a board setting. No PHI, no real financials.
# --------------------------------------------------------------------------- #

# Benefit realized per capability, expressed as a value-to-cost multiplier on the
# allocated Azure consumption. Grounded to the ROM BVA case (net benefit >> cost).
BENEFIT_MULTIPLIER: dict[str, float] = {
    "BMCA": 6.0,
    "OOA": 5.0,
    "DCA": 7.0,
    "ORSA": 4.5,
    "SBA": 4.0,
    "CSA": 3.5,
}
_DEFAULT_BENEFIT_MULTIPLIER = 4.0

# Decision cycles per CHF 1000 of allocated capability cost (synthetic, stable).
DECISION_CYCLES_PER_KCHF: dict[str, float] = {
    "BMCA": 120.0,
    "OOA": 90.0,
    "DCA": 80.0,
    "ORSA": 60.0,
    "SBA": 70.0,
    "CSA": 40.0,
}
_DEFAULT_DECISION_CYCLES = 75.0

EXEC_ROLES: tuple[tuple[str, str], ...] = (
    ("CEO", "Chief Executive Officer"),
    ("CFO", "Chief Financial Officer"),
    ("CIO", "Chief Information Officer"),
    ("COO", "Chief Operating Officer"),
    ("CTO", "Chief Technology Officer"),
    ("BOARD", "Board (shared summary)"),
)

# Map an Entra app role (persona ``app_role``) to the capability whose adoption
# it drives (design spec §4 capabilities). Governance / admin / read-only roles
# are intentionally unmapped — they do not count toward capability adoption.
DEFAULT_ROLE_CAPABILITY: dict[str, str] = {
    "HCC.BedManager": "BMCA",
    "HCC.FlowManager": "OOA",
    "HCC.OperationsLead": "OOA",
    "HCC.EDLead": "OOA",
    "HCC.DischargeCoordinator": "DCA",
    "HCC.ORCoordinator": "ORSA",
    "HCC.StaffingCoordinator": "SBA",
    "HCC.CrisisManager": "CSA",
}

# A successful Entra sign-in has resultType "0".
_SIGNIN_SUCCESS = "0"


def _num(value: object) -> float:
    try:
        return float(value)  # type: ignore[arg-type]
    except (TypeError, ValueError):
        return 0.0


# --------------------------------------------------------------------------- #
# Silver
# --------------------------------------------------------------------------- #

def to_silver(
    focus_rows: Iterable[Mapping],
    *,
    ingest_utc: str,
    source_seed: int,
) -> list[dict]:
    """Normalise Bronze FOCUS rows into Silver with keys + provenance.

    Adds foreign keys (``*_key``), a ``date_key`` / ``month_key`` and the
    provenance columns ``_ingest_utc`` / ``_source_seed`` (design spec §5). The
    FOCUS measure columns are preserved. Output is sorted for byte-stability.
    """
    out: list[dict] = []
    for row in focus_rows:
        out.append(
            {
                "date_key": row["ChargePeriodStart"],
                "month_key": row["BillingPeriod"],
                "service_key": row["ServiceName"],
                "meter_key": row["MeterName"],
                "resource_key": row["ResourceId"],
                "env_key": row["x_env"],
                "hospital_key": row["x_hospital"],
                "capability_key": row["x_capability"],
                "effective_cost": _num(row.get("EffectiveCost")),
                "billed_cost": _num(row.get("BilledCost")),
                "list_cost": _num(row.get("ListCost")),
                "quantity": _num(row.get("Quantity")),
                "currency": row.get("Currency", "CHF"),
                "_ingest_utc": ingest_utc,
                "_source_seed": int(source_seed),
            }
        )
    out.sort(key=lambda r: (r["date_key"], r["resource_key"], r["meter_key"]))
    return out


# --------------------------------------------------------------------------- #
# Gold dimensions
# --------------------------------------------------------------------------- #

def dim_service(focus_rows: Iterable[Mapping]) -> list[dict]:
    seen: dict[str, dict] = {}
    for row in focus_rows:
        key = row["ServiceName"]
        seen.setdefault(
            key,
            {
                "service_key": key,
                "service_name": key,
                "service_category": row["ServiceCategory"],
            },
        )
    return [seen[k] for k in sorted(seen)]


def dim_meter(focus_rows: Iterable[Mapping]) -> list[dict]:
    seen: dict[str, dict] = {}
    for row in focus_rows:
        key = row["MeterName"]
        seen.setdefault(
            key,
            {
                "meter_key": key,
                "meter_name": key,
                "meter_category": row["MeterCategory"],
                "meter_sub_category": row["MeterSubCategory"],
                "pricing_unit": row["PricingUnit"],
            },
        )
    return [seen[k] for k in sorted(seen)]


def dim_resource(focus_rows: Iterable[Mapping]) -> list[dict]:
    seen: dict[str, dict] = {}
    for row in focus_rows:
        key = row["ResourceId"]
        seen.setdefault(
            key,
            {
                "resource_key": key,
                "resource_name": row["ResourceName"],
                "resource_type": row["ResourceType"],
                "region": row["Region"],
                "service_key": row["ServiceName"],
                "env_key": row["x_env"],
                "hospital_key": row["x_hospital"],
                "capability_key": row["x_capability"],
            },
        )
    return [seen[k] for k in sorted(seen)]


def _distinct_dim(focus_rows: Iterable[Mapping], column: str, key_name: str) -> list[dict]:
    values = sorted({row[column] for row in focus_rows})
    return [{key_name: v} for v in values]


def dim_environment(focus_rows: Iterable[Mapping]) -> list[dict]:
    return _distinct_dim(focus_rows, "x_env", "env_key")


def dim_hospital(focus_rows: Iterable[Mapping]) -> list[dict]:
    return _distinct_dim(focus_rows, "x_hospital", "hospital_key")


def dim_capability(focus_rows: Iterable[Mapping]) -> list[dict]:
    return _distinct_dim(focus_rows, "x_capability", "capability_key")


def dim_date(focus_rows: Iterable[Mapping]) -> list[dict]:
    seen: dict[str, dict] = {}
    for row in focus_rows:
        key = row["ChargePeriodStart"]
        if key in seen:
            continue
        year, month, day = (int(p) for p in key.split("-"))
        seen[key] = {
            "date_key": key,
            "month_key": row["BillingPeriod"],
            "year": year,
            "month": month,
            "day": day,
        }
    return [seen[k] for k in sorted(seen)]


def dim_exec_role() -> list[dict]:
    return [{"exec_role_key": key, "exec_role_name": name} for key, name in EXEC_ROLES]


# --------------------------------------------------------------------------- #
# Gold facts
# --------------------------------------------------------------------------- #

def _round_costs(row: dict) -> dict:
    row["effective_cost"] = round(row["effective_cost"], 2)
    row["billed_cost"] = round(row["billed_cost"], 2)
    row["list_cost"] = round(row["list_cost"], 2)
    row["quantity"] = round(row["quantity"], 4)
    return row


def fact_azure_consumption(silver_rows: Iterable[Mapping]) -> list[dict]:
    """Aggregate Silver to resource × meter × day (design spec §5)."""
    groups: dict[tuple, dict] = {}
    for row in silver_rows:
        gk = (
            row["resource_key"],
            row["meter_key"],
            row["date_key"],
            row["env_key"],
            row["hospital_key"],
            row["capability_key"],
        )
        agg = groups.setdefault(
            gk,
            {
                "resource_key": gk[0],
                "meter_key": gk[1],
                "date_key": gk[2],
                "env_key": gk[3],
                "hospital_key": gk[4],
                "capability_key": gk[5],
                "effective_cost": 0.0,
                "billed_cost": 0.0,
                "list_cost": 0.0,
                "quantity": 0.0,
            },
        )
        agg["effective_cost"] += _num(row.get("effective_cost"))
        agg["billed_cost"] += _num(row.get("billed_cost"))
        agg["list_cost"] += _num(row.get("list_cost"))
        agg["quantity"] += _num(row.get("quantity"))
    out = [_round_costs(v) for v in groups.values()]
    out.sort(key=lambda r: (r["date_key"], r["resource_key"], r["meter_key"]))
    return out


def fact_budget(silver_rows: Iterable[Mapping]) -> list[dict]:
    """Plan baseline per env × capability × month (design spec §5).

    The plan is the **mean monthly actual** across the window for each
    (env, capability), giving each month a stable target so plan-vs-actual
    variance is realistic and non-trivial.
    """
    monthly: dict[tuple, float] = {}
    for row in silver_rows:
        gk = (row["env_key"], row["capability_key"], row["month_key"])
        monthly[gk] = monthly.get(gk, 0.0) + _num(row.get("effective_cost"))

    # Mean monthly actual per (env, capability).
    by_ec: dict[tuple, list[float]] = {}
    for (env, cap, _month), total in monthly.items():
        by_ec.setdefault((env, cap), []).append(total)
    plan_baseline = {ec: (sum(values) / len(values)) for ec, values in by_ec.items()}

    out = []
    for (env, cap, month), actual in monthly.items():
        plan = plan_baseline[(env, cap)]
        out.append(
            {
                "env_key": env,
                "capability_key": cap,
                "month_key": month,
                "plan_cost": round(plan, 2),
                "actual_cost": round(actual, 2),
                "variance_cost": round(actual - plan, 2),
            }
        )
    out.sort(key=lambda r: (r["month_key"], r["env_key"], r["capability_key"]))
    return out


def fact_value_realization(
    silver_rows: Iterable[Mapping],
    adoption_index: Mapping[tuple, int] | None = None,
) -> list[dict]:
    """Value realization per capability × month × hospital (design spec §5).

    ``adoption_index`` maps ``(capability_key, month_key, hospital_key) -> active
    user count``. When absent (T3 — before the Sprint 12 join in T4) the adoption
    count is ``0`` and ``benefit_realized`` is derived from the allocated cost and
    the synthetic :data:`BENEFIT_MULTIPLIER` alone. T4 supplies the index.
    """
    groups: dict[tuple, dict] = {}
    for row in silver_rows:
        gk = (row["capability_key"], row["month_key"], row["hospital_key"])
        agg = groups.setdefault(
            gk,
            {
                "capability_key": gk[0],
                "month_key": gk[1],
                "hospital_key": gk[2],
                "allocated_cost": 0.0,
            },
        )
        agg["allocated_cost"] += _num(row.get("effective_cost"))

    out = []
    for gk, agg in groups.items():
        cap = agg["capability_key"]
        allocated = agg["allocated_cost"]
        multiplier = BENEFIT_MULTIPLIER.get(cap, _DEFAULT_BENEFIT_MULTIPLIER)
        cycles_rate = DECISION_CYCLES_PER_KCHF.get(cap, _DEFAULT_DECISION_CYCLES)
        adoption = int(adoption_index.get(gk, 0)) if adoption_index else 0
        out.append(
            {
                "capability_key": cap,
                "month_key": agg["month_key"],
                "hospital_key": agg["hospital_key"],
                "allocated_cost": round(allocated, 2),
                "benefit_realized": round(allocated * multiplier, 2),
                "adoption_count": adoption,
                "decision_cycles": round(allocated / 1000.0 * cycles_rate, 1),
            }
        )
    out.sort(key=lambda r: (r["month_key"], r["capability_key"], r["hospital_key"]))
    return out


# --------------------------------------------------------------------------- #
# Adoption telemetry join (T4)
# --------------------------------------------------------------------------- #

def adoption_index_from_signins(
    signins: Iterable[Mapping],
    persona_hospital: Mapping[str, str] | None = None,
    role_capability: Mapping[str, str] | None = None,
) -> dict[tuple, int]:
    """Build the ``(capability, month, hospital) -> distinct active users`` index.

    Sprint 15 · T4. Consumes Sprint 12 sign-in rows (Bronze ``bva_adoption`` — or
    the 30-day synthetic backfill from ``adoption_seed_synthetic.py`` per design
    spec §14) and produces the adoption index consumed by
    :func:`fact_value_realization`.

    * Only **successful** sign-ins (``resultType == "0"``) count.
    * ``appRole`` is mapped to a capability via ``role_capability``
      (default :data:`DEFAULT_ROLE_CAPABILITY`); unmapped governance/admin roles
      are skipped.
    * The user's hospital comes from ``persona_hospital[upn]`` (built from
      ``personas.csv``); an unknown user falls back to ``Aggregated``.
    * The month comes from the ``YYYY-MM`` prefix of ``signInTimestamp``.
    * The value is the count of **distinct** ``upn`` in each group (active users).
    """
    role_map = role_capability or DEFAULT_ROLE_CAPABILITY
    hospitals = persona_hospital or {}

    groups: dict[tuple, set] = {}
    for row in signins:
        if str(row.get("resultType")) != _SIGNIN_SUCCESS:
            continue
        capability = role_map.get(row.get("appRole"))
        if not capability:
            continue
        timestamp = str(row.get("signInTimestamp", ""))
        if len(timestamp) < 7:
            continue
        month = timestamp[:7]
        upn = row.get("upn")
        if not upn:
            continue
        hospital = hospitals.get(upn, "Aggregated")
        groups.setdefault((capability, month, hospital), set()).add(upn)

    return {gk: len(users) for gk, users in groups.items()}
# --- END INLINED bva_transforms.py ---

# `T` alias mirrors the `import bva_transforms as T` in the source .py files
# so the medallion stage cells below stay identical to the extracted logic.
class _T:
    to_silver = staticmethod(to_silver)
    dim_service = staticmethod(dim_service)
    dim_meter = staticmethod(dim_meter)
    dim_resource = staticmethod(dim_resource)
    dim_environment = staticmethod(dim_environment)
    dim_hospital = staticmethod(dim_hospital)
    dim_capability = staticmethod(dim_capability)
    dim_date = staticmethod(dim_date)
    dim_exec_role = staticmethod(dim_exec_role)
    fact_azure_consumption = staticmethod(fact_azure_consumption)
    fact_budget = staticmethod(fact_budget)
    fact_value_realization = staticmethod(fact_value_realization)
    adoption_index_from_signins = staticmethod(adoption_index_from_signins)
T = _T


## 3. Silver (FOCUS → Silver contract)


In [ ]:
# Mirrors data-platform/notebooks/bva/build_silver_bva.py.
from datetime import datetime, timezone

_bronze = spark.read.table('bronze.bva_consumption')
_focus_rows = [r.asDict() for r in _bronze.collect()]

_source_seed = 0
if _focus_rows and 'x_source_seed' in _focus_rows[0]:
    _source_seed = int(_focus_rows[0]['x_source_seed'])

_silver = T.to_silver(
    _focus_rows,
    ingest_utc=datetime.now(timezone.utc).isoformat(),
    source_seed=_source_seed,
)
_silver_df = spark.createDataFrame(_silver)
(
    _silver_df.write.format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('silver.bva_consumption')
)
print(f'silver: wrote silver.bva_consumption ({_silver_df.count()} rows)')


## 4. Gold dims (8 dimension tables)


In [ ]:
# Mirrors data-platform/notebooks/bva/build_gold_bva_dims.py.
def _write_gold(rows, table):
    df = spark.createDataFrame(rows)
    (
        df.write.format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(f'gold.bva_{table}')
    )
    print(f'gold: wrote gold.bva_{table} ({df.count()} rows)')

_silver_for_dims = spark.read.table('silver.bva_consumption')
_focus_rows_for_dims = [r.asDict() for r in spark.read.table('bronze.bva_consumption').collect()]

_write_gold(T.dim_service(_focus_rows_for_dims), 'dim_service')
_write_gold(T.dim_meter(_focus_rows_for_dims), 'dim_meter')
_write_gold(T.dim_resource(_focus_rows_for_dims), 'dim_resource')
_write_gold(T.dim_environment(_focus_rows_for_dims), 'dim_environment')
_write_gold(T.dim_hospital(_focus_rows_for_dims), 'dim_hospital')
_write_gold(T.dim_capability(_focus_rows_for_dims), 'dim_capability')
_write_gold(T.dim_date(_focus_rows_for_dims), 'dim_date')
_write_gold(T.dim_exec_role(), 'dim_exec_role')
print(f'silver source rows: {_silver_for_dims.count()}')


## 5. Gold facts (3 fact tables)


In [ ]:
# Mirrors data-platform/notebooks/bva/build_gold_bva_facts.py.
def _persona_hospital():
    try:
        rows = [r.asDict() for r in spark.read.table('gold.dim_persona').collect()]
    except Exception:  # noqa: BLE001 - persona dim may not yet exist
        return {}
    mapping = {}
    for r in rows:
        upn = r.get('upn') or r.get('user_principal_name')
        hospital = r.get('default_hospital') or r.get('hospital_key')
        if upn and hospital:
            mapping[upn] = hospital
    return mapping


def _load_adoption_index():
    try:
        signins = [r.asDict() for r in spark.read.table('bronze.bva_adoption').collect()]
    except Exception:  # noqa: BLE001 - table not present until Sprint 12 adoption Bronze joins
        return None
    return T.adoption_index_from_signins(signins, persona_hospital=_persona_hospital())


_silver_for_facts = [r.asDict() for r in spark.read.table('silver.bva_consumption').collect()]

_write_gold(T.fact_azure_consumption(_silver_for_facts), 'fact_azure_consumption')
_write_gold(T.fact_budget(_silver_for_facts), 'fact_budget')

_adoption_index = _load_adoption_index()
_write_gold(
    T.fact_value_realization(_silver_for_facts, adoption_index=_adoption_index),
    'fact_value_realization',
)
print('gold facts complete.')
